In [2]:
# ============================================================
# PHASE 07 V2 — SETUP
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd

SEED = 42

# ------------------------------------------------------------
# PROJECT ROOT
# ------------------------------------------------------------

PROJECT_ROOT = Path.cwd().parent

if not (PROJECT_ROOT / "cric_cervix").exists():
    PROJECT_ROOT = Path(
        r"C:\Users\nanda\Documents\cervical-semi-sl"
    )

# ------------------------------------------------------------
# EXISTING PHASE 06 DATA
# ------------------------------------------------------------

PROCESSED_ROOT = (
    PROJECT_ROOT / "data" / "processed"
)

CRIC_UNITS_PATH = (
    PROCESSED_ROOT /
    "cric_learning_units.csv"
)

RIVA_UNITS_PATH = (
    PROCESSED_ROOT /
    "riva_learning_units.csv"
)

# ------------------------------------------------------------
# NEW V2 SPLIT OUTPUTS
# ------------------------------------------------------------

SPLIT_ROOT = (
    PROJECT_ROOT / "data" / "splits_v2"
)

CRIC_SPLIT_PATH = (
    SPLIT_ROOT /
    "cric_splits_v2.csv"
)

RIVA_SPLIT_PATH = (
    SPLIT_ROOT /
    "riva_splits_v2.csv"
)

SPLIT_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

# ------------------------------------------------------------
# FINAL V2 PROTOCOL
# ------------------------------------------------------------

TRAIN_FRAC = 0.70
VAL_FRAC = 0.10
TEST_FRAC = 0.20

assert (
    TRAIN_FRAC
    + VAL_FRAC
    + TEST_FRAC
    == 1.0
)

# ------------------------------------------------------------
# CHECK EXISTING MANIFESTS
# ------------------------------------------------------------

assert CRIC_UNITS_PATH.exists(), (
    f"CRIC manifest not found:\n{CRIC_UNITS_PATH}"
)

assert RIVA_UNITS_PATH.exists(), (
    f"RIVA manifest not found:\n{RIVA_UNITS_PATH}"
)

# ------------------------------------------------------------
# LOAD EXISTING PHASE 06 MANIFESTS
# ------------------------------------------------------------

cric = pd.read_csv(
    CRIC_UNITS_PATH
)

riva = pd.read_csv(
    RIVA_UNITS_PATH
)

print("=" * 60)
print("PHASE 07 V2 SETUP")
print("=" * 60)

print("\nExisting CRIC manifest:")
print(CRIC_UNITS_PATH)
print("Shape:", cric.shape)

print("\nExisting RIVA manifest:")
print(RIVA_UNITS_PATH)
print("Shape:", riva.shape)

print("\nV2 split output:")
print(SPLIT_ROOT)

print("\nProtocol:")
print("Train      :", TRAIN_FRAC)
print("Validation :", VAL_FRAC)
print("Test       :", TEST_FRAC)
print("Seed       :", SEED)

print("\nPHASE 07 V2 setup PASSED.")

PHASE 07 V2 SETUP

Existing CRIC manifest:
c:\Users\nanda\Documents\cervical-semi-sl\data\processed\cric_learning_units.csv
Shape: (11534, 9)

Existing RIVA manifest:
c:\Users\nanda\Documents\cervical-semi-sl\data\processed\riva_learning_units.csv
Shape: (15949, 12)

V2 split output:
c:\Users\nanda\Documents\cervical-semi-sl\data\splits_v2

Protocol:
Train      : 0.7
Validation : 0.1
Test       : 0.2
Seed       : 42

PHASE 07 V2 setup PASSED.


In [3]:
# ============================================================
# CELL 2 — INSPECT EXISTING MANIFESTS
# ============================================================

print("=" * 60)
print("CRIC MANIFEST")
print("=" * 60)

print("\nColumns:")
print(cric.columns.tolist())

print("\nFirst 3 rows:")
display(cric.head(3))


print("\n" + "=" * 60)
print("RIVA MANIFEST")
print("=" * 60)

print("\nColumns:")
print(riva.columns.tolist())

print("\nFirst 3 rows:")
display(riva.head(3))

CRIC MANIFEST

Columns:
['dataset', 'image_filename', 'cell_id', 'x_pixel', 'y_pixel', 'class_bethesda', 'binary_label', 'crop_path', 'crop_valid']

First 3 rows:


,dataset,image_filename,cell_id,x_pixel,y_pixel,class_bethesda,binary_label,crop_path,crop_valid
0,CRIC,9ae8a4edde40219bad6303cebc672ee4.png,1,792,462,SCC,1,data\processed\cric_crops\9ae8a4edde40219bad63...,True
1,CRIC,9ae8a4edde40219bad6303cebc672ee4.png,2,601,678,SCC,1,data\processed\cric_crops\9ae8a4edde40219bad63...,True
2,CRIC,9ae8a4edde40219bad6303cebc672ee4.png,3,363,467,SCC,1,data\processed\cric_crops\9ae8a4edde40219bad63...,True



RIVA MANIFEST

Columns:
['dataset', 'image_filename', 'x_norm', 'y_norm', 'class_bethesda', 'class_annotated', 'cluster_idx', 'binary_label', 'x_pixel', 'y_pixel', 'crop_path', 'crop_valid']

First 3 rows:


,dataset,image_filename,x_norm,y_norm,class_bethesda,class_annotated,cluster_idx,binary_label,x_pixel,y_pixel,crop_path,crop_valid
0,RIVA,LSIL_45_3.png,1000.549618,526.330789,ASCUS,ASCUS,40000.0,1,1001,526,data\processed\riva_crops\LSIL_45_3_cluster_40...,True
1,RIVA,LSIL_45_3.png,974.493639,534.147583,ASCUS,ASCUS,40001.0,1,974,534,data\processed\riva_crops\LSIL_45_3_cluster_40...,True
2,RIVA,LSIL_45_3.png,849.424936,138.096692,ASCUS,ASCUS,40002.0,1,849,138,data\processed\riva_crops\LSIL_45_3_cluster_40...,True


In [4]:
# ============================================================
# CELL 3 — DEFINE FINAL V2 GROUPS
# ============================================================

# ------------------------------------------------------------
# CRIC
# Grouping unit = parent image
# ------------------------------------------------------------

CRIC_GROUP_COLUMN = "image_filename"


# ------------------------------------------------------------
# RIVA
# Grouping unit = parent slide
# Filename format:
# category_sample_patch.png
#
# Example:
# LSIL_45_3.png -> LSIL_45
# ------------------------------------------------------------

def get_riva_slide_id(filename):

    stem = Path(filename).stem
    parts = stem.split("_")

    assert len(parts) >= 3, (
        f"Unexpected RIVA filename: {filename}"
    )

    return "_".join(parts[:-1])


riva["slide_id"] = (
    riva["image_filename"]
    .apply(get_riva_slide_id)
)

RIVA_GROUP_COLUMN = "slide_id"


# ------------------------------------------------------------
# VALIDATE
# ------------------------------------------------------------

assert cric[CRIC_GROUP_COLUMN].nunique() == 400
assert riva[RIVA_GROUP_COLUMN].nunique() == 111

assert len(cric) == 11534
assert len(riva) == 15949

assert cric["binary_label"].isin([0, 1]).all()
assert riva["binary_label"].isin([0, 1]).all()


# Every CRIC image contains at least one learning unit
assert (
    cric.groupby(CRIC_GROUP_COLUMN)
    .size()
    .min()
    > 0
)

# Every RIVA slide contains at least one learning unit
assert (
    riva.groupby(RIVA_GROUP_COLUMN)
    .size()
    .min()
    > 0
)


print("=" * 60)
print("V2 GROUPING STRUCTURE")
print("=" * 60)

print("\nCRIC")
print("Learning units :", len(cric))
print("Images         :", cric[CRIC_GROUP_COLUMN].nunique())
print("Grouping       : image level")

print("\nRIVA")
print("Learning units :", len(riva))
print("Images         :", riva["image_filename"].nunique())
print("Slides         :", riva[RIVA_GROUP_COLUMN].nunique())
print("Grouping       : slide level")

print("\nRIVA example:")
print(
    riva[
        ["image_filename", "slide_id"]
    ].head()
)

print("\nV2 grouping validation PASSED.")

V2 GROUPING STRUCTURE

CRIC
Learning units : 11534
Images         : 400
Grouping       : image level

RIVA
Learning units : 15949
Images         : 959
Slides         : 111
Grouping       : slide level

RIVA example:
  image_filename slide_id
0  LSIL_45_3.png  LSIL_45
1  LSIL_45_3.png  LSIL_45
2  LSIL_45_3.png  LSIL_45
3  LSIL_45_3.png  LSIL_45
4  LSIL_45_3.png  LSIL_45

V2 grouping validation PASSED.


In [5]:
# ============================================================
# CELL 4 — CREATE FINAL 70/10/20 GROUP-AWARE SPLITS
# ============================================================

def create_group_aware_split(
    dataframe,
    group_column,
    seed=42,
    train_frac=0.70,
    val_frac=0.10,
    test_frac=0.20
):
    rng = np.random.default_rng(seed)

    # --------------------------------------------------------
    # Build one row per group
    # --------------------------------------------------------

    groups = (
        dataframe
        .groupby(group_column)
        .agg(
            n_units=("binary_label", "size"),
            abnormal_units=("binary_label", "sum")
        )
        .reset_index()
    )

    groups["abnormal_rate"] = (
        groups["abnormal_units"]
        / groups["n_units"]
    )

    # Random value for reproducible tie-breaking
    groups["random"] = rng.random(
        len(groups)
    )

    # Process larger groups first
    groups = groups.sort_values(
        ["n_units", "random"],
        ascending=[False, True]
    ).reset_index(drop=True)

    # --------------------------------------------------------
    # Target number of learning units
    # --------------------------------------------------------

    total_units = len(dataframe)

    targets = {
        "train": train_frac * total_units,
        "validation": val_frac * total_units,
        "test": test_frac * total_units
    }

    assigned_units = {
        "train": 0,
        "validation": 0,
        "test": 0
    }

    assigned_groups = {
        "train": [],
        "validation": [],
        "test": []
    }

    # --------------------------------------------------------
    # Assign complete groups
    # --------------------------------------------------------

    for _, row in groups.iterrows():

        group_name = row[group_column]
        group_size = int(row["n_units"])

        remaining_capacity = {
            split:
            targets[split] - assigned_units[split]
            for split in targets
        }

        # Prefer the split with the largest
        # remaining capacity.
        chosen_split = max(
            remaining_capacity,
            key=remaining_capacity.get
        )

        assigned_groups[chosen_split].append(
            group_name
        )

        assigned_units[chosen_split] += group_size

    # --------------------------------------------------------
    # Convert group assignments to sample-level split
    # --------------------------------------------------------

    group_to_split = {}

    for split, group_list in assigned_groups.items():

        for group_name in group_list:
            group_to_split[group_name] = split

    result = dataframe.copy()

    result["split"] = (
        result[group_column]
        .map(group_to_split)
    )

    # --------------------------------------------------------
    # Safety checks
    # --------------------------------------------------------

    assert result["split"].notna().all()

    group_leakage = (
        result
        .groupby(group_column)["split"]
        .nunique()
    )

    assert group_leakage.max() == 1

    return result


# ------------------------------------------------------------
# CRIC — IMAGE LEVEL
# ------------------------------------------------------------

cric_split = create_group_aware_split(
    cric,
    group_column=CRIC_GROUP_COLUMN,
    seed=SEED,
    train_frac=TRAIN_FRAC,
    val_frac=VAL_FRAC,
    test_frac=TEST_FRAC
)


# ------------------------------------------------------------
# RIVA — SLIDE LEVEL
# ------------------------------------------------------------

riva_split = create_group_aware_split(
    riva,
    group_column=RIVA_GROUP_COLUMN,
    seed=SEED,
    train_frac=TRAIN_FRAC,
    val_frac=VAL_FRAC,
    test_frac=TEST_FRAC
)


# ------------------------------------------------------------
# BASIC OUTPUT
# ------------------------------------------------------------

print("=" * 60)
print("FINAL V2 SPLITS CREATED")
print("=" * 60)

print("\nCRIC learning units:")
print(
    cric_split["split"]
    .value_counts()
    .reindex(
        ["train", "validation", "test"]
    )
)

print("\nRIVA learning units:")
print(
    riva_split["split"]
    .value_counts()
    .reindex(
        ["train", "validation", "test"]
    )
)

FINAL V2 SPLITS CREATED

CRIC learning units:
split
train         8074
validation    1153
test          2307
Name: count, dtype: int64

RIVA learning units:
split
train         11171
validation     1582
test           3196
Name: count, dtype: int64


In [7]:
# ============================================================
# CELL 5 — FINAL VALIDATION + SAVE V2 SPLITS
# ============================================================

def validate_final_split(
    dataframe,
    group_column,
    dataset_name
):

    print("\n" + "=" * 60)
    print(dataset_name)
    print("=" * 60)

    # --------------------------------------------------------
    # Required splits
    # --------------------------------------------------------

    expected_splits = {
        "train",
        "validation",
        "test"
    }

    assert set(
        dataframe["split"].unique()
    ) == expected_splits

    # --------------------------------------------------------
    # Group leakage check
    # --------------------------------------------------------

    group_split_counts = (
        dataframe
        .groupby(group_column)["split"]
        .nunique()
    )

    assert group_split_counts.max() == 1

    # --------------------------------------------------------
    # Learning-unit counts
    # --------------------------------------------------------

    counts = (
        dataframe["split"]
        .value_counts()
        .reindex(
            ["train", "validation", "test"]
        )
    )

    print("\nLearning units:")
    print(counts)

    # --------------------------------------------------------
    # Group counts
    # --------------------------------------------------------

    group_counts = (
        dataframe
        .groupby("split")[group_column]
        .nunique()
        .reindex(
            ["train", "validation", "test"]
        )
    )

    print("\nGroups:")
    print(group_counts)

    # --------------------------------------------------------
    # Class distribution
    # --------------------------------------------------------

    class_counts = (
        dataframe
        .groupby(
            ["split", "binary_label"]
        )
        .size()
        .unstack(
            fill_value=0
        )
        .reindex(
            index=[
                "train",
                "validation",
                "test"
            ],
            columns=[0, 1],
            fill_value=0
        )
    )

    print("\nClass counts:")
    print(class_counts)

    # --------------------------------------------------------
    # Abnormal proportion
    # --------------------------------------------------------

    abnormal_rate = (
        dataframe
        .groupby("split")["binary_label"]
        .mean()
        .reindex(
            ["train", "validation", "test"]
        )
    )

    print("\nAbnormal proportion:")
    print(abnormal_rate.round(4))

    print("\nGroup leakage: NONE")


# ------------------------------------------------------------
# VALIDATE
# ------------------------------------------------------------

validate_final_split(
    cric_split,
    CRIC_GROUP_COLUMN,
    "CRIC — IMAGE-LEVEL SPLIT"
)

validate_final_split(
    riva_split,
    RIVA_GROUP_COLUMN,
    "RIVA — SLIDE-LEVEL SPLIT"
)


# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------

cric_split.to_csv(
    CRIC_SPLIT_PATH,
    index=False
)

riva_split.to_csv(
    RIVA_SPLIT_PATH,
    index=False
)


# ------------------------------------------------------------
# VERIFY SAVED FILES
# ------------------------------------------------------------

assert CRIC_SPLIT_PATH.exists()
assert RIVA_SPLIT_PATH.exists()

cric_saved = pd.read_csv(
    CRIC_SPLIT_PATH
)

riva_saved = pd.read_csv(
    RIVA_SPLIT_PATH
)

assert len(cric_saved) == 11534
assert len(riva_saved) == 15949

assert (
    cric_saved
    .groupby(CRIC_GROUP_COLUMN)["split"]
    .nunique()
    .max()
    == 1
)

assert (
    riva_saved
    .groupby(RIVA_GROUP_COLUMN)["split"]
    .nunique()
    .max()
    == 1
)


print("\n" + "=" * 70)
print("PHASE 07 V2 — COMPLETE")
print("=" * 70)

print("\nProtocol:")
print("  Train      : 70%")
print("  Validation : 10%")
print("  Test       : 20%")
print("  Seed       :", SEED)

print("\nGrouping:")
print("  CRIC : image level")
print("  RIVA : slide level")

print("\nSaved:")
print(CRIC_SPLIT_PATH)
print(RIVA_SPLIT_PATH)

print("\nAll final V2 split checks PASSED.")


CRIC — IMAGE-LEVEL SPLIT

Learning units:
split
train         8074
validation    1153
test          2307
Name: count, dtype: int64

Groups:
split
train         209
validation     78
test          113
Name: image_filename, dtype: int64

Class counts:
binary_label     0     1
split                   
train         4482  3592
validation     694   459
test          1603   704

Abnormal proportion:
split
train         0.4449
validation    0.3981
test          0.3052
Name: binary_label, dtype: float64

Group leakage: NONE

RIVA — SLIDE-LEVEL SPLIT

Learning units:
split
train         11171
validation     1582
test           3196
Name: count, dtype: int64

Groups:
split
train         53
validation    24
test          34
Name: slide_id, dtype: int64

Class counts:
binary_label     0     1
split                   
train         7332  3839
validation    1115   467
test          2070  1126

Abnormal proportion:
split
train         0.3437
validation    0.2952
test          0.3523
Name: binary_lab